In [ ]:
# 1/4 - clone main + prototype (fresh, fault-tolerant)
import subprocess, os
print('===== NATIVE FlashAttentionFwdLse TEST (PURE TKV, scalar GEMM + custom exp) =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 2/4 - build native .so + .tkv C .so. NO C injection. CUNG flags.
import subprocess, os
FLAGS='-O2 -mavx2 -mfma'
root='/content/TVmain/Release/tools/simd_native'
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
p=subprocess.run('cd '+root+' && gcc '+FLAGS+' -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('native build rc=',p.returncode)
if p.returncode!=0: print(p.stderr[:2000])
ex='/content/TVcpu/Release/cuda_backend/examples'
cc=ex+'/flash_fwd_lse_cpu.cpu.c'; so=ex+'/flash_fwd_lse_cpu.so'
txt=open(cc).read()
print('C INJECTED =', 'tkv_flash' in txt, '(expect False)')
print('buffer malloc/free =', 'malloc(' in txt and 'free(' in txt, ' custom exp rint =', 'rintf(' in txt)
p2=subprocess.run('gcc '+FLAGS+' -shared -fPIC '+cc+' -o '+so+' -lm -lpthread -ldl', shell=True, capture_output=True, text=True, timeout=600)
print('tkv C build rc=',p2.returncode)
if p2.returncode!=0: print(p2.stderr[:2500])

In [ ]:
# 3/4 - compare FlashAttentionFwdLse (ctx + lse)
import ctypes, numpy as np
P=ctypes.POINTER(ctypes.c_float); I=ctypes.c_int; D=ctypes.c_double
args=[P,I,P,I,P,I,I,I,I,I,D]
native=ctypes.CDLL('/content/TVmain/Release/tools/simd_native/tkv_simd_native.so').FlashAttentionFwdLse
native.argtypes=args; native.restype=I
tkv=ctypes.CDLL('/content/TVcpu/Release/cuda_backend/examples/flash_fwd_lse_cpu.so').FlashAttentionFwdLse
tkv.argtypes=args; tkv.restype=I
rng=np.random.default_rng(53)
for n,h,head in [(64,256,0),(128,512,3),(256,1024,7),(65,192,1)]:
    qkv=rng.standard_normal(n*3*h).astype(np.float32)
    ca=np.zeros(n*64,np.float32); cc=np.zeros(n*64,np.float32)
    la=np.zeros(n,np.float32); lc=np.zeros(n,np.float32)
    inv=0.125
    native(qkv.ctypes.data_as(P),qkv.size,ca.ctypes.data_as(P),ca.size,la.ctypes.data_as(P),la.size,n,h,64,head,inv)
    tkv  (qkv.ctypes.data_as(P),qkv.size,cc.ctypes.data_as(P),cc.size,lc.ctypes.data_as(P),lc.size,n,h,64,head,inv)
    dc=np.count_nonzero(ca.view(np.uint32)!=cc.view(np.uint32))
    dl=np.count_nonzero(la.view(np.uint32)!=lc.view(np.uint32))
    print('n=%d h=%d head=%d ctx_bitwise=%s ctx_diff=%d/%d lse_diff=%d/%d ctx_maxabs=%.3g lse_maxabs=%.3g'%(
        n,h,head,dc==0,dc,ca.size,dl,la.size,np.abs(ca-cc).max(),np.abs(la-lc).max()))

In [ ]:
# 4/4 - summary
print('Flash path has s0.125 tolerance waiver. Report ctx/lse maxabs; bitwise diff is informational.')